# Customer & Sales Analytics with PySpark

**Pipeline:** Raw Transactions → Cleaning → Sales KPI → Customer 360 → RFM → Segmentation → Retention & Pareto → Business Insights

## 1. Mục tiêu 
- Dùng PySpark DataFrame để xử lý dữ liệu giao dịch lớn.
- Xây dựng KPI bán hàng và khách hàng.
- Phân tích theo thời gian, sản phẩm, kênh, vùng.
- Xây Customer 360 và RFM.
- Phân khúc khách hàng bằng K-Means.
- Phân tích repeat purchase, retention và Pareto 80/20.
- Chuyển kết quả thành business insights.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('TH04_CustomerSales').master('local[*]').getOrCreate()
print('Spark:', spark.version)

## 2. Đọc dữ liệu

In [ ]:
sales = spark.read.option('header', True).option('inferSchema', True).csv('data/sales_transactions_dirty.csv')
customers = spark.read.option('header', True).option('inferSchema', True).csv('data/customers.csv')
products = spark.read.option('header', True).option('inferSchema', True).csv('data/products.csv')
print('sales =', sales.count(), 'customers =', customers.count(), 'products =', products.count())
sales.printSchema()
sales.show(5, truncate=False)

### Bài tập: Data Understanding
1. Xác định grain của từng bảng.
2. Đếm số khách hàng, sản phẩm, đơn hàng và giao dịch.
3. Cho biết khoảng thời gian dữ liệu.
4. Giải thích vì sao số dòng giao dịch lớn hơn số đơn hàng.

In [ ]:
sales.select(F.min('Date').alias('MinDate'), F.max('Date').alias('MaxDate'), F.countDistinct('OrderID').alias('Orders'), F.countDistinct('CustomerID').alias('Customers'), F.countDistinct('ProductID').alias('Products')).show()

## 3. Data Quality Audit

In [ ]:
sales.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in sales.columns]).show(truncate=False)
print('Duplicate rows:', sales.count() - sales.dropDuplicates().count())

### Bài tập
Kiểm tra:
- Missing value.
- Duplicate.
- Quantity/UnitPrice <= 0.
- Discount ngoài [0,1].
- Revenue/Profit bất thường.

Đề xuất quy tắc cleaning và giải thích tác động đến KPI.

In [ ]:
sales_clean = (sales.dropDuplicates()
    .dropna(subset=['CustomerID','ProductID','Revenue'])
    .filter((F.col('Quantity') > 0) & (F.col('UnitPrice') > 0))
    .filter((F.col('Discount') >= 0) & (F.col('Discount') <= 1))
    .withColumn('Date', F.to_date('Date')))
print('Before:', sales.count(), 'After:', sales_clean.count())

## 4. Sales KPI

In [ ]:
kpi = sales_clean.agg(
    F.sum('Revenue').alias('Revenue'),
    F.sum('Profit').alias('Profit'),
    F.sum('Quantity').alias('UnitsSold'),
    F.countDistinct('OrderID').alias('Orders'),
    F.countDistinct('CustomerID').alias('Customers'))
kpi.show(truncate=False)

### Bài tập
Tính thêm:
- Average Order Value (AOV)
- Profit Margin
- Revenue per Customer
- Items per Order
- Return Rate

Giải thích ý nghĩa kinh doanh của từng KPI.

In [ ]:
order_level = sales_clean.groupBy('OrderID').agg(F.sum('Revenue').alias('OrderRevenue'),F.sum('Quantity').alias('Items'),F.max('IsReturn').alias('HasReturn'))
order_level.agg(F.avg('OrderRevenue').alias('AOV'),F.avg('Items').alias('ItemsPerOrder'),F.avg('HasReturn').alias('ReturnRate')).show()

## 5. Sales Trend Analytics

In [ ]:
monthly = (sales_clean.withColumn('YearMonth',F.date_format('Date','yyyy-MM'))
    .groupBy('YearMonth').agg(F.sum('Revenue').alias('Revenue'),F.sum('Profit').alias('Profit'),F.countDistinct('OrderID').alias('Orders'))
    .orderBy('YearMonth'))
monthly.show(36)

### Bài tập: Câu hỏi kinh doanh
1. Tháng nào doanh thu cao nhất/thấp nhất?
2. Có seasonality không?
3. Doanh thu tăng có đi cùng lợi nhuận tăng không?
4. Tính growth MoM bằng Window + lag().

In [ ]:
w=Window.orderBy('YearMonth')
monthly_growth=(monthly.withColumn('PrevRevenue',F.lag('Revenue').over(w))
    .withColumn('MoM_Growth', (F.col('Revenue')-F.col('PrevRevenue'))/F.col('PrevRevenue')))
monthly_growth.show(36)

## 6. Product, Category, Channel & Region Analytics

In [ ]:
sales_enriched=(sales_clean
    .join(products.select('ProductID','ProductName','Category'),'ProductID','left')
    .join(customers.select('CustomerID','CustomerSegment','Region'),'CustomerID','left'))

category_perf=(sales_enriched.groupBy('Category').agg(F.sum('Revenue').alias('Revenue'),F.sum('Profit').alias('Profit'),F.sum('Quantity').alias('Units')).withColumn('Margin',F.col('Profit')/F.col('Revenue')).orderBy(F.desc('Revenue')))
category_perf.show()

### Bài tập
- Top 10 sản phẩm theo Revenue và Profit.
- Category có Revenue cao nhưng Margin thấp.
- Channel có AOV cao nhất.
- Region nào có doanh thu và số khách hàng cao nhất.
- So sánh CustomerSegment theo Revenue/Profit/AOV.

## 7. Customer 360

In [ ]:
customer360=(sales_enriched.groupBy('CustomerID')
    .agg(F.min('Date').alias('FirstPurchase'),F.max('Date').alias('LastPurchase'),F.countDistinct('OrderID').alias('Orders'),F.sum('Quantity').alias('Units'),F.sum('Revenue').alias('Revenue'),F.sum('Profit').alias('Profit'),F.countDistinct('ProductID').alias('DistinctProducts'))
    .withColumn('AOV',F.col('Revenue')/F.col('Orders')))
customer360.orderBy(F.desc('Revenue')).show(10,truncate=False)

### Bài tập
1. Top 1%, 5%, 10% khách hàng đóng góp bao nhiêu % doanh thu?
2. Khách hàng mua nhiều nhất có phải khách hàng sinh lợi nhất không?
3. Tìm khách hàng có Revenue cao nhưng Profit thấp/âm.

## 8. Repeat Purchase & Purchase Interval

In [ ]:
repeat_stats=(customer360.withColumn('IsRepeat',(F.col('Orders')>=2).cast('int')).agg(F.avg('IsRepeat').alias('RepeatCustomerRate'),F.avg('Orders').alias('AvgOrdersPerCustomer')))
repeat_stats.show()

order_dates=sales_clean.select('CustomerID','OrderID','Date').dropDuplicates(['CustomerID','OrderID'])
w=Window.partitionBy('CustomerID').orderBy('Date')
intervals=(order_dates.withColumn('PrevDate',F.lag('Date').over(w)).withColumn('DaysBetween',F.datediff('Date','PrevDate')))
intervals.select(F.avg('DaysBetween').alias('AvgDaysBetweenPurchases'),F.expr('percentile_approx(DaysBetween,0.5)').alias('MedianDays')).show()

## 9. RFM Analysis

In [ ]:
max_date=sales_clean.agg(F.max('Date')).first()[0]
rfm=(sales_clean.groupBy('CustomerID').agg(
    F.datediff(F.lit(max_date),F.max('Date')).alias('Recency'),
    F.countDistinct('OrderID').alias('Frequency'),
    F.sum('Revenue').alias('Monetary')))
rfm.describe().show()

### Bài tập: RFM Scoring
Tạo điểm R, F, M từ 1–5 bằng `ntile(5)` hoặc quantile. Quy ước Recency thấp là tốt, Frequency/Monetary cao là tốt. Sau đó gán các nhóm: Champions, Loyal, Potential, At Risk, Lost/Other.

In [ ]:
wr=Window.orderBy(F.desc('Recency'))
wf=Window.orderBy('Frequency')
wm=Window.orderBy('Monetary')
rfm_scored=(rfm.withColumn('R',F.ntile(5).over(wr)).withColumn('F',F.ntile(5).over(wf)).withColumn('M',F.ntile(5).over(wm)))
rfm_scored=(rfm_scored.withColumn('Segment',F.when((F.col('R')>=4)&(F.col('F')>=4)&(F.col('M')>=4),'Champions').when((F.col('R')>=3)&(F.col('F')>=3),'Loyal').when((F.col('R')>=4)&(F.col('F')<=2),'Potential').when((F.col('R')<=2)&(F.col('F')>=3),'At Risk').otherwise('Lost/Other')))
rfm_scored.groupBy('Segment').agg(F.count('*').alias('Customers'),F.avg('Monetary').alias('AvgMonetary')).orderBy(F.desc('AvgMonetary')).show()

## 10. Customer Segmentation bằng K-Means

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

assembler=VectorAssembler(inputCols=['Recency','Frequency','Monetary'],outputCol='rawFeatures')
scaled=StandardScaler(inputCol='rawFeatures',outputCol='features',withMean=True,withStd=True).fit(assembler.transform(rfm)).transform(assembler.transform(rfm))
evaluator=ClusteringEvaluator(featuresCol='features')
results=[]
for k in range(2,7):
    model=KMeans(k=k,seed=42,featuresCol='features').fit(scaled)
    pred=model.transform(scaled)
    results.append((k,evaluator.evaluate(pred)))
print(results)

### Bài tập
Chọn k phù hợp dựa trên Silhouette Score, sau đó profile từng cluster theo Recency, Frequency, Monetary. Đặt tên business-friendly cho mỗi cluster và đề xuất hành động marketing.

## 11. Pareto 80/20

In [ ]:
cust_rev=customer360.select('CustomerID','Revenue').orderBy(F.desc('Revenue'))
w_all=Window.orderBy(F.desc('Revenue')).rowsBetween(Window.unboundedPreceding,Window.currentRow)
total=customer360.agg(F.sum('Revenue')).first()[0]
pareto=(cust_rev.withColumn('CumRevenue',F.sum('Revenue').over(w_all)).withColumn('CumShare',F.col('CumRevenue')/F.lit(total)))
pareto.show(20)

### Bài tập
Xác định tỷ lệ khách hàng cần để tạo ra 50%, 80%, 90% doanh thu. Lặp lại phân tích cho sản phẩm.

## 12. Cohort & Retention

In [ ]:
orders=sales_clean.select('CustomerID','OrderID','Date').dropDuplicates(['CustomerID','OrderID'])
first=orders.groupBy('CustomerID').agg(F.min('Date').alias('FirstDate'))
cohort=(orders.join(first,'CustomerID').withColumn('CohortMonth',F.date_format('FirstDate','yyyy-MM')).withColumn('OrderMonth',F.date_format('Date','yyyy-MM')).withColumn('MonthIndex',F.months_between(F.trunc('Date','month'),F.trunc('FirstDate','month')).cast('int')))
retention=cohort.groupBy('CohortMonth','MonthIndex').agg(F.countDistinct('CustomerID').alias('Customers')).orderBy('CohortMonth','MonthIndex')
retention.show(30)

## 13. Spark SQL Challenge

In [ ]:
sales_enriched.createOrReplaceTempView('sales')

Viết ít nhất 6 truy vấn Spark SQL:
1. Revenue theo tháng.
2. Top 10 sản phẩm theo Revenue.
3. Profit Margin theo Category.
4. Revenue/AOV theo Channel.
5. Top khách hàng.
6. CustomerSegment theo Revenue và Profit.

## 14. Business Insight Challenge
Từ kết quả trên, trả lời tối thiểu 5 câu hỏi:
- Nguồn tăng trưởng chính đến từ đâu?
- Nhóm khách hàng nào nên ưu tiên giữ chân?
- Có phụ thuộc quá mức vào một số khách hàng/sản phẩm không?
- Kênh nào hiệu quả về doanh thu nhưng kém về margin?
- Hành động nào có thể tăng repeat purchase/AOV/profit?

## 15. Executive Summary
Viết tối đa 1 trang gồm:
- 5 insight định lượng.
- 3 recommendation.
- 2 hạn chế của dữ liệu/phân tích.
- Mỗi insight phải trích số liệu tính được từ notebook, không suy đoán.

## 16. Sản phẩm nộp
- Notebook `.ipynb` đã chạy.
- Báo cáo 2-4 trang hoặc slide.
- Bảng RFM/segmentation.
- 5 insight + 3 recommendation.
- README ngắn nếu có thay đổi code/dataset.